In [0]:
dbutils.widgets.text(
    "Environment",
    "dev",
    "Nom du catalogue"
)

env = dbutils.widgets.get("Environment")

print(
    f"Starting Serverless streaming integration test on {env}"
)

### Paramètre d'environnement

Le widget `Environment` permet de choisir le catalogue utilisé pour le test.

Dans notre cas :

`Environment = dev`

La valeur est stockée dans `env` et sera transmise aux différents notebooks et au Workflow Databricks.

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/02-setup

In [0]:
SH = SetupHelper(env)

SH.cleanup()

print("Environment cleaned.")

### Nettoyage initial

Le test commence sur un environnement propre afin que les résultats précédents ne faussent pas les nouvelles validations.

`cleanup → nouveau test propre`

La landing zone Azure `raw` doit également être vide avant de recommencer un test complet.

In [0]:
#### Netoyage syplementaire pour eviter les erreurs 

conf = Config()

raw_dir = conf.base_dir_data + "/raw"
checkpoint_root = conf.base_dir_checkpoint + "/checkpoints"

dbutils.fs.rm(raw_dir, True)
dbutils.fs.rm(checkpoint_root, True)

dbutils.fs.mkdirs(raw_dir + "/registered_users_bz")
dbutils.fs.mkdirs(raw_dir + "/gym_logins_bz")
dbutils.fs.mkdirs(raw_dir + "/kafka_multiplex_bz")

print("RAW et checkpoints remis à zéro.")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

print(
    f"Connected to Databricks workspace: {w.config.host}"
)

### Connexion au Workspace

`WorkspaceClient()` permet de piloter Databricks depuis Python.

Il utilise l'authentification du notebook actuel.

Nous n'avons donc pas besoin de stocker manuellement un `Host` et un `AccessToken`, ce qui est plus sécurisé.

In [0]:
from databricks.sdk.service.jobs import (
    Task,
    NotebookTask,
    Source
)

notebook_path = (
    "/Workspace/Users/"
    "fayelatyr61@gmail.com/"
    "azure-databricks-realtime-health-platform/"
    "07-run"
)

job = w.jobs.create(
    name="stream-test-serverless",

    tasks=[
        Task(
            task_key="stream-test-task",

            notebook_task=NotebookTask(
                notebook_path=notebook_path,
                source=Source.WORKSPACE
            )
        )
    ]
)

job_id = job.job_id

print(
    f"Serverless Job created: {job_id}"
)

### Création du Workflow Serverless

Cette partie crée automatiquement un Job Databricks contenant une tâche Notebook.

Le Job exécute :

`07-run`

Aucun cluster n'est défini dans le code.

Databricks utilise donc le calcul Serverless disponible dans le Workspace.

Le `job_id` permet ensuite de déclencher, surveiller puis supprimer ce Job.

In [0]:
run = w.jobs.run_now(
    job_id=job_id,

    notebook_params={
        "Environment": env,
        "RunType": "once",
        "ProcessingTime": "5 seconds"
    }
)

initial_run_id = run.run_id

print(
    f"Initial Job run started: {initial_run_id}"
)

### Premier lancement du Workflow

Le premier lancement exécute `07-run` sur Serverless.

`RunType = once` utilise notre traitement `availableNow` :

`traiter les données disponibles → puis s'arrêter`

Cette première exécution sert principalement à préparer les tables et les données historiques avant le test.

In [0]:
import time

while True:

    run_info = w.jobs.get_run(
        run_id=initial_run_id
    )

    state = run_info.state.life_cycle_state.value

    print(
        f"Initial run state: {state}"
    )

    if state in [
        "TERMINATED",
        "SKIPPED",
        "INTERNAL_ERROR"
    ]:
        break

    time.sleep(20)

print(
    "Initial Serverless run completed."
)

### Attente de l'initialisation

Le notebook vérifie régulièrement l'état du Job.

Il attend que l'exécution initiale soit terminée avant de produire les données de test.

Cela évite de commencer le test alors que les tables ne sont pas encore prêtes.

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/03-history-loader

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/10-producer

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/04-bronze

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/05-silver

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/06-gold

### Chargement des composants de test

Ces notebooks chargent les classes nécessaires pour produire puis valider les données :

`HistoryLoader`

`Producer`

`Bronze`

`Silver`

`Gold`

Elles seront utilisées pour vérifier l'ensemble du pipeline après chaque Payload.

In [0]:
HL = HistoryLoader(env)
PR = Producer()

BZ = Bronze(env)
SL = Silver(env)
GL = Gold(env)

SH.validate()
HL.validate()

print(
    "Setup and historical data validated."
)

### Validation de l'environnement

Avant de produire le premier Payload, on vérifie que :

- les tables du projet existent ;
- le Setup est correct ;
- `date_lookup` et les données historiques sont disponibles.

Le test des données métier peut ensuite commencer.

In [0]:
conf = Config()

raw_dir = conf.base_dir_data + "/raw"

print("Reset de la landing zone avant Payload 1...")

# Supprime complètement les anciennes données de test
dbutils.fs.rm(
    raw_dir + "/registered_users_bz",
    True
)

dbutils.fs.rm(
    raw_dir + "/gym_logins_bz",
    True
)

dbutils.fs.rm(
    raw_dir + "/kafka_multiplex_bz",
    True
)

# Recrée les dossiers vides
dbutils.fs.mkdirs(
    raw_dir + "/registered_users_bz"
)

dbutils.fs.mkdirs(
    raw_dir + "/gym_logins_bz"
)

dbutils.fs.mkdirs(
    raw_dir + "/kafka_multiplex_bz"
)

print("Landing zone vide et prête pour Payload 1.")

In [0]:
PR.produce(1)

PR.validate(1)

### Production du Payload 1

Le Producer copie le premier jeu de données depuis :

`test_data`

vers :

`raw`

Cela simule l'arrivée d'un premier lot de données dans le système.

In [0]:
run1 = w.jobs.run_now(
    job_id=job_id,

    notebook_params={
        "Environment": env,
        "RunType": "once",
        "ProcessingTime": "5 seconds"
    }
)

run1_id = run1.run_id

print(
    f"Payload 1 processing started: {run1_id}"
)

### Traitement du Payload 1

Le même Job Serverless est relancé.

`07-run` exécute alors :

`raw → Bronze → Silver → Gold`

Grâce aux checkpoints, seules les nouvelles données doivent être prises en compte.

In [0]:
import time

while True:

    info = w.jobs.get_run(
        run_id=run1_id
    )

    state = info.state.life_cycle_state.value

    print(
        f"Payload 1 run: {state}"
    )

    if state in [
        "TERMINATED",
        "SKIPPED",
        "INTERNAL_ERROR"
    ]:
        break

    time.sleep(20)

print(
    "Payload 1 run completed."
)

### Attente du traitement

On attend la fin du pipeline avant de vérifier les résultats.

Cela garantit que Bronze, Silver et Gold ont terminé le traitement du premier Payload.

In [0]:
BZ.validate(1)

SL.validate(1)

GL.validate(1)

print(
    "Payload 1 validated successfully."
)

### Validation du Payload 1

Les trois couches sont contrôlées :

`Bronze → données ingérées`

`Silver → données nettoyées et transformées`

`Gold → résultats métier`

Le Payload 1 est considéré comme correct seulement si toutes les validations passent.

In [0]:
PR.produce(2)

PR.validate(2)

### Production du Payload 2

Le deuxième Payload représente de nouvelles données arrivant après le premier traitement.

Cette étape permet de tester le comportement incrémental du pipeline.

In [0]:
run2 = w.jobs.run_now(
    job_id=job_id,

    notebook_params={
        "Environment": env,
        "RunType": "once",
        "ProcessingTime": "5 seconds"
    }
)

run2_id = run2.run_id

print(
    f"Payload 2 processing started: {run2_id}"
)

### Traitement du Payload 2

Le même Workflow Serverless traite maintenant le deuxième lot.

Le système doit conserver le résultat du premier traitement tout en intégrant uniquement les nouvelles données.

Cette étape teste notamment :

`checkpoints + MERGE + CDC + déduplication`

In [0]:
import time

while True:

    info = w.jobs.get_run(
        run_id=run2_id
    )

    state = info.state.life_cycle_state.value

    print(
        f"Payload 2 run: {state}"
    )

    if state in [
        "TERMINATED",
        "SKIPPED",
        "INTERNAL_ERROR"
    ]:
        break

    time.sleep(20)

print(
    "Payload 2 run completed."
)

In [0]:
BZ.validate(2)

SL.validate(2)

GL.validate(2)

print(
    "Payload 2 validated successfully."
)

### Validation finale

Cette validation vérifie l'état du Lakehouse après les deux Payloads.

Le résultat final doit correspondre à :

`Payload 1 + Payload 2`

sans doublons ni perte de données.

In [0]:
w.jobs.delete(
    job_id=job_id
)

print(
    f"Test Job {job_id} deleted."
)

### Nettoyage du Workflow

Le Job créé uniquement pour le test est supprimé après les validations.

Cela évite de laisser inutilement des Jobs de test dans Databricks.

Les tables peuvent être conservées temporairement afin d'inspecter les résultats.

# Test d'intégration Serverless

Ce notebook teste le comportement incrémental du pipeline Databricks sur un environnement Serverless.

Le scénario suit la logique du test streaming du cours tout en tenant compte des limitations du calcul Serverless.

Le flux est :

`Cleanup`

`↓`

`Création d'un Workflow Serverless`

`↓`

`Initialisation Setup + History`

`↓`

`Payload 1`

`↓`

`07-run → Bronze → Silver → Gold`

`↓`

`Validation`

`↓`

`Payload 2`

`↓`

`07-run → traitement incrémental`

`↓`

`Validation finale`

`↓`

`Suppression du Workflow`

Le Job Serverless est créé sans définir de cluster : Databricks gère automatiquement les ressources de calcul.